# HanziGen - 字型生成训练 (Google Colab)

## 使用前准备

**只需上传目标字体到 Google Drive 根目录**（MyDrive/）：
- `your_font.ttf` 或 `your_font.otf` — 你的目标字体
- 参考字体 Jigmo 会自动从官网下载，无需手动上传

**运行前**：菜单 → 代码执行程序 → 更改运行时类型 → **T4 GPU**

---

## 断连容错说明（重要）

本项目已内置**断连自动恢复**机制（基于 `--resume_from` 参数）：

- **阶段状态文件** `colab_state.json` 记录每个阶段是否完成，断连后重开 Notebook 会自动跳过已完成阶段。
- **周期性检查点**：训练期间每 `CKPT_SAVE_INTERVAL` 个 epoch 自动备份**完整训练状态**（模型+优化器+调度器+epoch）到本地 `checkpoints/`。
- **真正 epoch 级精确续训**：Cell 1 检测到已有 checkpoint 时，会自动把 `train_*.sh` 的 `RESUME_FROM` 指向最新权重，重跑训练 Cell 即从断点 epoch 精确继续（恢复模型/优化器/调度器/学习率）。
- **Drive 双备份**：本地 `/content` 断连即丢，所有成果（checkpoints / data / charsets / samples）都镜像到 Drive。

---

## 流程

```
Cell 0: 配置字体名 + 阶段开关
Cell 1: 挂载Drive + Clone + 装依赖 + 复制字体 + 下载Jigmo + 改写脚本 + 断连自检 + 验证GPU
Cell 2: 数据准备（分析字体 → 生成数据集 → 提取字集）【可断连恢复】
Cell 3: 训练 VQ-VAE（自动精确续训）
Cell 4: 训练 LDM（自动精确续训）
Cell 5: 推理 + 指标 + 转 SVG
```

---
## Cell 0: 配置参数

> **只改这里！** 填你上传到 Google Drive 的字体文件名（英文）。

In [ ]:
# ==================== 修改你的字体文件名 ====================
TARGET_FONT = "your_font.otf"    # 改成你上传到 Drive 的字体名
# ==========================================================

# ============ 改成你自己的 fork 仓库地址（用于 git clone） ============
GIT_REPO = "https://github.com/ICW-k/HanziGen_ICWfork.git"   # 改成你的 fork
# ====================================================================

FONT_NAME = TARGET_FONT.rsplit(".", 1)[0]
DRIVE = "/content/drive/MyDrive"
PROJECT = "/content/HanziGen"

# ==================== 训练阶段开关（默认全开）====================
# 置 False 可跳过某阶段。断连恢复时无需手动设置，会自动跳过已完成阶段。
DO_DATA_PREP = True       # Cell 2: 数据准备
DO_TRAIN_VQVAE = True     # Cell 3: 训练 VQ-VAE
DO_TRAIN_LDM = True       # Cell 4: 训练 LDM
DO_INFERENCE = True       # Cell 5: 推理+指标+转SVG
# ==========================================================

# 检查点周期性备份间隔（epoch 数）与 Drive 同步目录
CKPT_SAVE_INTERVAL = 5    # 每 N 个 epoch 自动备份完整训练状态
DRIVE_BACKUP_DIR = f"{DRIVE}/HanziGen_Backup"
DRIVE_RESULTS_DIR = f"{DRIVE}/HanziGen_Results"
STATE_FILE = "colab_state.json"

print(f"目标字体: {TARGET_FONT}")
print(f"字体名称: {FONT_NAME}")
print(f"Git 仓库: {GIT_REPO}")
print(f"阶段开关: 数据准备={DO_DATA_PREP} VQVAE={DO_TRAIN_VQVAE} LDM={DO_TRAIN_LDM} 推理={DO_INFERENCE}")

---
## Cell 1: 全自动环境初始化 + 断连自检

> 挂载 Drive → Clone 你的 fork → 安装依赖 → 复制字体 → 下载 Jigmo → 改写脚本 → **断连自检** → 验证 GPU

In [ ]:
import os, shutil, json, re, sys

# ===== 1. 挂载 Google Drive =====
from google.colab import drive
drive.mount('/content/drive')

# ===== 2. Clone 你的 fork =====
if not os.path.exists(PROJECT):
    !git clone -q {GIT_REPO} {PROJECT}
os.chdir(PROJECT)
print(f"工作目录: {os.getcwd()}")
!ls

# ===== 3. 安装依赖 =====
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q -r requirements.txt
print("\n依赖安装完成！")

# ===== 4. 检查并复制字体 =====
drive_font = f"{DRIVE}/{TARGET_FONT}"
if not os.path.exists(drive_font):
    raise FileNotFoundError(
        f"字体文件不存在: {drive_font}\n"
        f"请确保已将 {TARGET_FONT} 上传到 Google Drive 根目录"
    )

os.makedirs("fonts/jigmo", exist_ok=True)
shutil.copy2(drive_font, f"fonts/{TARGET_FONT}")
print(f"字体已复制: {TARGET_FONT}")

# ===== 5. 把字体路径写进所有 .sh 脚本 =====
import glob
for sh_file in glob.glob("scripts/*.sh"):
    with open(sh_file, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r'fonts/[\w.-]+\.(ttf|otf)', f'fonts/{TARGET_FONT}', content)
    with open(sh_file, "w", encoding="utf-8") as f:
        f.write(content)
print("脚本字体路径已统一替换")

# ===== 6. Jigmo 参考字体：一律从官方 ZIP 下载 =====
import zipfile, io, urllib.request
from fontTools.ttLib import TTFont

jigmo_files = ["jigmo.ttf", "jigmo2.ttf", "jigmo3.ttf"]

def download_jigmo_fonts(target_dir="fonts/jigmo"):
    os.makedirs(target_dir, exist_ok=True)
    zip_url = "https://kamichikoichi.github.io/jigmo/Jigmo-20250912.zip"
    print(f"  下载 Jigmo ZIP: {zip_url}")
    try:
        resp = urllib.request.urlopen(zip_url, timeout=30)
        data = resp.read()
        if len(data) < 10000:
            raise ValueError(f"下载数据太小 ({len(data)} bytes)")
        zf = zipfile.ZipFile(io.BytesIO(data))
    except Exception as e:
        print(f"  [ERROR] ZIP 下载失败: {e}")
        return False
    for zip_name in zf.namelist():
        basename = os.path.basename(zip_name).lower()
        if basename in jigmo_files:
            zf.extract(zip_name, target_dir)
            extracted = os.path.join(target_dir, zip_name)
            target = os.path.join(target_dir, basename)
            if extracted != target:
                if os.path.exists(target):
                    os.remove(target)
                os.rename(extracted, target)
    zf.close()
    return True

def validate_font_file(fpath):
    try:
        f = TTFont(fpath)
        if "cmap" not in f:
            return False, "缺少 cmap 表"
        return True, f"OK ({len(f.getBestCmap())} glyphs)"
    except Exception as e:
        return False, str(e)[:80]

need_download = False
for fname in jigmo_files:
    fpath = f"fonts/jigmo/{fname}"
    if os.path.exists(fpath):
        valid, msg = validate_font_file(fpath)
        if not valid:
            print(f"  [WARN] {fname} 无效 ({msg})")
            os.remove(fpath)
            need_download = True
    else:
        need_download = True

if need_download:
    if not download_jigmo_fonts():
        raise RuntimeError("Jigmo 下载失败！")
    print("Jigmo 字体下载完成，验证中...")
    for fname in jigmo_files:
        valid, msg = validate_font_file(f"fonts/jigmo/{fname}")
        print(f"    [{'OK' if valid else 'ERROR'}] {fname}: {msg}")
        if not valid:
            raise RuntimeError(f"Jigmo 字体 {fname} 验证失败！")

# ===== 7. 断连自检 + 自动续训配置 =====
os.makedirs("checkpoints", exist_ok=True)

def load_state() -> dict:
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_state(state: dict) -> None:
    with open(STATE_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

state = load_state()
state.setdefault("font", FONT_NAME)
save_state(state)

vqvae_ckpt = f"checkpoints/vqvae_{FONT_NAME}.pth"
ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"
have_vqvae = os.path.exists(vqvae_ckpt)
have_ldm = os.path.exists(ldm_ckpt)

print("\n===== 断连自检 =====")
print(f"  数据准备 (data/): {'已完成' if os.path.isdir('data') else '未完成'}")
print(f"  VQ-VAE 检查点:   {'存在: '+vqvae_ckpt if have_vqvae else '不存在'}")
print(f"  LDM 检查点:      {'存在: '+ldm_ckpt if have_ldm else '不存在'}")

# 自动精确续训：若存在检查点，把 train_*.sh 的 RESUME_FROM="" 改写为实际路径
def _set_resume_from(sh_path: str, ckpt: str) -> None:
    with open(sh_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r'RESUME_FROM="[^"]*"', f'RESUME_FROM="{ckpt}"', content)
    with open(sh_path, "w", encoding="utf-8") as f:
        f.write(content)

if have_vqvae:
    _set_resume_from("scripts/train_vqvae.sh", vqvae_ckpt)
    print(f"  [续训] train_vqvae.sh 的 RESUME_FROM -> {vqvae_ckpt}")

if have_ldm:
    _set_resume_from("scripts/train_ldm.sh", ldm_ckpt)
    print(f"  [续训] train_ldm.sh 的 RESUME_FROM -> {ldm_ckpt}")

if not have_vqvae and not os.path.isdir("data") and DO_DATA_PREP:
    print("  提示: 需先运行 Cell 2 完成数据准备")

# ===== 8. 验证 GPU =====
import torch
assert torch.cuda.is_available(), "GPU 不可用！请检查：菜单 → 代码执行程序 → 更改运行时类型 → T4 GPU"
prop = torch.cuda.get_device_properties(0)
print(f"\nGPU: {torch.cuda.get_device_name(0)}")
print(f"显存: {prop.total_memory / 1024**3:.1f} GB")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")
print(f"\n全部初始化完成！字体: fonts/{TARGET_FONT}")

---
## Cell 2: 数据准备（约 10 分钟，断连可恢复）

> 分析字体覆盖率 → 渲染字型图片 → 提取训练/验证字符集
>
> **断连恢复**：重新运行本 Cell 会先检查 `data/` 与字符集是否已存在，已存在则自动跳过。

In [ ]:
import os, json; os.chdir(PROJECT)

if not DO_DATA_PREP:
    print("DO_DATA_PREP=False，跳过 Cell 2")
else:
    def _load_state() -> dict:
        if os.path.exists(STATE_FILE):
            try:
                with open(STATE_FILE, "r", encoding="utf-8") as f:
                    return json.load(f)
            except Exception:
                return {}
        return {}

    def _save_state(s: dict) -> None:
        with open(STATE_FILE, "w", encoding="utf-8") as f:
            json.dump(s, f, ensure_ascii=False, indent=2)

    state = _load_state()

    data_done = os.path.isdir("data/reference") and os.path.isdir("data/target")
    splits_done = os.path.exists(f"charsets/splits/{FONT_NAME}/train.txt") and \
                 os.path.exists(f"charsets/splits/{FONT_NAME}/val.txt")

    if state.get("data_prep_done") and data_done and splits_done:
        print("数据准备已完成，跳过 Cell 2")
    else:
        print("\n===== 1. 分析字体覆盖率 =====")
        !bash scripts/analyze_font.sh

        print("\n===== 2. 生成数据集图片 =====")
        !bash scripts/prepare_dataset.sh

        print("\n===== 3. 提取训练/验证字符集 =====")
        !bash scripts/extract_charset.sh

        if not (os.path.isdir("data/reference") and os.path.isdir("data/target")):
            raise RuntimeError("data/ 目录生成失败，请检查 prepare_dataset.sh")
        if not os.path.exists(f"charsets/splits/{FONT_NAME}/train.txt"):
            raise RuntimeError("train.txt 未生成，请检查 extract_charset.sh")

        state["data_prep_done"] = True
        _save_state(state)
        print("\n===== 数据准备完成并已记录状态 =====")

    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    for folder in ["charsets", "checkpoints"]:
        if os.path.isdir(folder):
            !cp -r "{folder}/" "{DRIVE_BACKUP_DIR}/" 2>/dev/null
            print(f"备份 {folder}/ -> Drive 完成")

---
## Cell 3: 训练 VQ-VAE（约 6-8 小时，自动精确续训）

> **断连恢复**：先运行 Cell 0、Cell 1（会自检并自动把 `train_vqvae.sh` 的 `RESUME_FROM` 指向最新权重），再运行本 Cell 即从断点 epoch 精确续训。
>
> 若 `DO_TRAIN_VQVAE=False` 则跳过。

In [ ]:
import os; os.chdir(PROJECT)

if not DO_TRAIN_VQVAE:
    print("DO_TRAIN_VQVAE=False，跳过 Cell 3")
elif not os.path.isdir("data"):
    print("data/ 不存在，请先运行 Cell 2 完成数据准备")
else:
    !bash scripts/train_vqvae.sh

### 💾 训练间隙手动备份（可选，随时可运行）

> 训练仍在进行时可跳过；用于在长训练间隙手动把最新成果同步到 Drive，防意外丢失。

In [ ]:
import os; os.chdir(PROJECT)
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
for folder in ["checkpoints", "charsets", "data", "samples_" + FONT_NAME]:
    if os.path.exists(folder):
        !cp -r "{folder}/" "{DRIVE_BACKUP_DIR}/" 2>/dev/null
        print(f"备份 {folder}/ 完成")
print("手动备份完成！")

---
## Cell 4: 训练 LDM（约 10-15 小时，自动精确续训）

> 依赖 VQ-VAE 训练完成（需要 `checkpoints/vqvae_{FONT_NAME}.pth` 存在）
>
> **断连恢复**：先运行 Cell 0、Cell 1，再运行本 Cell 即从断点 epoch 精确续训。
>
> 若 `DO_TRAIN_LDM=False` 则跳过。

In [ ]:
import os; os.chdir(PROJECT)

vqvae_ckpt = f"checkpoints/vqvae_{FONT_NAME}.pth"

if not DO_TRAIN_LDM:
    print("DO_TRAIN_LDM=False，跳过 Cell 4")
elif not os.path.exists(vqvae_ckpt):
    print(f"{vqvae_ckpt} 不存在，请先完成 Cell 3 训练 VQ-VAE")
else:
    print("VQ-VAE 检查点确认：")
    !ls -lh checkpoints/
    !bash scripts/train_ldm.sh

---
## Cell 5: 推理生成 + 指标 + 转换 SVG

> 依赖 LDM 训练完成（`checkpoints/ldm_{FONT_NAME}.pth`）
>
> 若 `DO_INFERENCE=False` 则跳过。

In [ ]:
import os; os.chdir(PROJECT)

ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"

if not DO_INFERENCE:
    print("DO_INFERENCE=False，跳过 Cell 5")
elif not os.path.exists(ldm_ckpt):
    print(f"{ldm_ckpt} 不存在，请先完成 Cell 4 训练 LDM")
else:
    !bash scripts/inference.sh
    !bash scripts/compute_metrics.sh
    !bash scripts/convert_to_svg.sh

### 📦 最终结果保存到 Google Drive

In [ ]:
import os; os.chdir(PROJECT)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
for folder in ["checkpoints", "charsets", "samples_" + FONT_NAME]:
    if os.path.exists(folder):
        !cp -r "{folder}/" "{DRIVE_RESULTS_DIR}/" 2>/dev/null
        print(f"结果 {folder}/ 已保存到 Drive")
print(f"\n结果已保存到: {DRIVE_RESULTS_DIR}/")
!ls -lh "{DRIVE_RESULTS_DIR}/checkpoints/"

---
## 断连恢复指南（唯一操作流程）

Colab 断连后重开 Notebook，**只需按顺序运行**：

1. **Cell 0**（恢复变量）
2. **Cell 1**（重新初始化 + 断连自检 + **自动改写续训参数**）
3. **依次重新运行之后的 Cell 2 / 3 / 4 / 5**
   - 已完成的阶段会自动跳过（通过 `colab_state.json` 与产物存在性判断）
   - 未完成的训练阶段（Cell 3 / 4）会自动从最近 checkpoint 的断点 epoch **精确续训**

> **无需手动编辑脚本**，Cell 1 已自动处理 `RESUME_FROM`。

## 从 Drive 备份恢复（在全新 /content 但保留 Drive 时使用）

> 若 `/content` 已被清空、仅剩 Drive 备份，先在 Cell 1 后运行本 Cell 从 Drive 拉回检查点与数据。

In [ ]:
# 从 Drive 拉回 checkpoint 与数据（可选，在 Cell 1 之后运行）
import os; os.chdir(PROJECT)

for folder in ["checkpoints", "charsets", "data", "samples_" + FONT_NAME]:
    src = f"{DRIVE_BACKUP_DIR}/{folder}"
    if os.path.isdir(src) and os.listdir(src):
        !cp -r "{src}/"*. "{folder}/" 2>/dev/null
        print(f"{folder} 已从 Drive 恢复")
    else:
        print(f"{folder} 无备份，需重新生成")

print("\n恢复完成！继续按顺序运行后续 Cell 即可（Cell 1 已自动配置续训）")

---
## 防断连脚本（可选）

> 在训练 Cell 之前运行，每 10 分钟自动保活。

In [ ]:
from IPython.display import Javascript
Javascript("""
function keepAlive() {
    var btn = document.querySelector('colab-connect-button');
    if (btn) btn.click();
    setTimeout(keepAlive, 600000);
}
keepAlive();
""")
print("防断连已启动（每 10 分钟自动保活）")

---
## 常见问题

| 问题 | 解决 |
|------|------|
| `fonts/xxx.ttf` 不存在 | Cell 0 填的字体名和 Drive 里的文件名是否一致？ |
| `charsets/.../covered.txt` 不存在 | Cell 2 的 `analyze_font.sh` 是否成功？ |
| `data/reference` 不是目录 | Cell 2 的 `prepare_dataset.sh` 是否成功？ |
| 显存不足 (OOM) | 改小训练脚本中的 `BATCH_SIZE`（VQVAE 用 8，LDM 用 16 时 T4 建议各减半） |
| 字体名必须英文 | 如 `my_font.otf`，不要中文 |
| 断连后重跑训练从头开始 | 确保 Cell 1 已把 `RESUME_FROM` 指向 checkpoint，且 `checkpoints/` 存在对应 `.pth` |
| 续训后想重头训练 | 删除 `checkpoints/*.pth` 与 `colab_state.json`，再重跑 Cell 1 |